In [ ]:
%%writefile data_utils.py
"""
Data utilities for Federated Learning
Handles CIFAR-10/MNIST loading and IID/non-IID splits
"""

import numpy as np
import torch
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
from collections import defaultdict


def load_dataset(dataset_name='cifar10', data_path='./data'):
    """
    Load CIFAR-10 or MNIST dataset

    Args:
        dataset_name: 'cifar10' or 'mnist'
        data_path: path to store/load data

    Returns:
        train_dataset, test_dataset
    """
    if dataset_name.lower() == 'cifar10':
        transform_train = transforms.Compose([
            transforms.RandomCrop(32, padding=4),
            transforms.RandomHorizontalFlip(),
            transforms.ToTensor(),
            transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
        ])
        transform_test = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
        ])

        train_dataset = datasets.CIFAR10(root=data_path, train=True, download=True, transform=transform_train)
        test_dataset = datasets.CIFAR10(root=data_path, train=False, download=True, transform=transform_test)

    elif dataset_name.lower() == 'mnist':
        transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize((0.1307,), (0.3081,))
        ])

        train_dataset = datasets.MNIST(root=data_path, train=True, download=True, transform=transform)
        test_dataset = datasets.MNIST(root=data_path, train=False, download=True, transform=transform)

    else:
        raise ValueError(f"Dataset {dataset_name} not supported. Use 'cifar10' or 'mnist'")

    return train_dataset, test_dataset


def create_iid_split(dataset, num_clients):
    """
    Split dataset evenly (IID) across clients

    Args:
        dataset: PyTorch dataset
        num_clients: number of clients

    Returns:
        dict: {client_id: list of indices}
    """
    num_samples = len(dataset)
    indices = np.random.permutation(num_samples)
    split_size = num_samples // num_clients

    client_indices = {}
    for i in range(num_clients):
        start_idx = i * split_size
        end_idx = start_idx + split_size if i < num_clients - 1 else num_samples
        client_indices[i] = indices[start_idx:end_idx].tolist()

    return client_indices


def create_dirichlet_split(dataset, num_clients, alpha, min_samples_per_client=10):
    """
    Create non-IID split using Dirichlet distribution

    Args:
        dataset: PyTorch dataset
        num_clients: number of clients
        alpha: concentration parameter (smaller = more skewed)
        min_samples_per_client: minimum samples each client should have

    Returns:
        dict: {client_id: list of indices}
    """
    # Get labels
    if hasattr(dataset, 'targets'):
        labels = np.array(dataset.targets)
    elif hasattr(dataset, 'labels'):
        labels = np.array(dataset.labels)
    else:
        labels = np.array([dataset[i][1] for i in range(len(dataset))])

    num_classes = len(np.unique(labels))

    # Group indices by class
    class_indices = defaultdict(list)
    for idx, label in enumerate(labels):
        class_indices[label].append(idx)

    # Initialize client indices
    client_indices = {i: [] for i in range(num_clients)}

    # For each class, sample proportions from Dirichlet and distribute
    for class_id in range(num_classes):
        indices = np.array(class_indices[class_id])
        np.random.shuffle(indices)

        # Sample proportions from Dirichlet
        proportions = np.random.dirichlet(np.repeat(alpha, num_clients))
        proportions = proportions / proportions.sum()  # Normalize

        # Calculate number of samples per client for this class
        proportions = (np.cumsum(proportions) * len(indices)).astype(int)[:-1]

        # Split indices according to proportions
        splits = np.split(indices, proportions)

        # Assign to clients
        for client_id, split in enumerate(splits):
            client_indices[client_id].extend(split.tolist())

    # Ensure minimum samples per client
    for client_id in range(num_clients):
        if len(client_indices[client_id]) < min_samples_per_client:
            print(f"Warning: Client {client_id} has only {len(client_indices[client_id])} samples")

    return client_indices


def get_dataloaders(dataset, client_indices, batch_size=32, shuffle=True):
    """
    Create DataLoader for each client

    Args:
        dataset: PyTorch dataset
        client_indices: dict of {client_id: list of indices}
        batch_size: batch size for training
        shuffle: whether to shuffle data

    Returns:
        dict: {client_id: DataLoader}
    """
    client_loaders = {}
    for client_id, indices in client_indices.items():
        subset = Subset(dataset, indices)
        loader = DataLoader(subset, batch_size=batch_size, shuffle=shuffle, num_workers=2)
        client_loaders[client_id] = loader

    return client_loaders


def analyze_data_distribution(dataset, client_indices, num_classes=10):
    """
    Analyze and print the label distribution for each client

    Args:
        dataset: PyTorch dataset
        client_indices: dict of {client_id: list of indices}
        num_classes: number of classes
    """
    if hasattr(dataset, 'targets'):
        labels = np.array(dataset.targets)
    elif hasattr(dataset, 'labels'):
        labels = np.array(dataset.labels)
    else:
        labels = np.array([dataset[i][1] for i in range(len(dataset))])

    print("\n" + "="*80)
    print("DATA DISTRIBUTION ANALYSIS")
    print("="*80)

    for client_id, indices in client_indices.items():
        client_labels = labels[indices]
        class_counts = np.bincount(client_labels, minlength=num_classes)

        print(f"\nClient {client_id} (Total: {len(indices)} samples):")
        print(f"  Class distribution: {class_counts.tolist()}")
        print(f"  Percentages: {(class_counts / len(indices) * 100).round(2).tolist()}")

    print("="*80 + "\n")

Writing data_utils.py


In [ ]:
%%writefile models.py
"""
Model architectures for Federated Learning
"""

import torch
import torch.nn as nn
import torch.nn.functional as F


class SimpleCNN(nn.Module):
    """
    Simple CNN for CIFAR-10
    Architecture: 2 Conv layers + 2 FC layers
    """
    def __init__(self, num_classes=10, input_channels=3):
        super(SimpleCNN, self).__init__()

        # Convolutional layers
        self.conv1 = nn.Conv2d(input_channels, 32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)

        # Pooling
        self.pool = nn.MaxPool2d(2, 2)

        # Fully connected layers
        # After 2 pooling layers: 32x32 -> 16x16 -> 8x8
        self.fc1 = nn.Linear(64 * 8 * 8, 128)
        self.fc2 = nn.Linear(128, num_classes)

        # Dropout for regularization
        self.dropout = nn.Dropout(0.25)

    def forward(self, x):
        # Conv block 1
        x = self.pool(F.relu(self.conv1(x)))

        # Conv block 2
        x = self.pool(F.relu(self.conv2(x)))

        # Flatten
        x = x.view(-1, 64 * 8 * 8)

        # FC layers
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)

        return x


class SimpleMNISTCNN(nn.Module):
    """
    Simple CNN for MNIST
    Architecture: 2 Conv layers + 2 FC layers
    """
    def __init__(self, num_classes=10):
        super(SimpleMNISTCNN, self).__init__()

        # Convolutional layers
        self.conv1 = nn.Conv2d(1, 16, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)

        # Pooling
        self.pool = nn.MaxPool2d(2, 2)

        # Fully connected layers
        # After 2 pooling layers: 28x28 -> 14x14 -> 7x7
        self.fc1 = nn.Linear(32 * 7 * 7, 64)
        self.fc2 = nn.Linear(64, num_classes)

        # Dropout
        self.dropout = nn.Dropout(0.25)

    def forward(self, x):
        # Conv block 1
        x = self.pool(F.relu(self.conv1(x)))

        # Conv block 2
        x = self.pool(F.relu(self.conv2(x)))

        # Flatten
        x = x.view(-1, 32 * 7 * 7)

        # FC layers
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)

        return x


def get_model(model_name='simplecnn', num_classes=10, dataset='cifar10'):
    """
    Factory function to get model

    Args:
        model_name: name of the model
        num_classes: number of output classes
        dataset: dataset name ('cifar10' or 'mnist')

    Returns:
        PyTorch model
    """
    if model_name.lower() == 'simplecnn':
        if dataset.lower() == 'mnist':
            return SimpleMNISTCNN(num_classes=num_classes)
        else:
            return SimpleCNN(num_classes=num_classes, input_channels=3)
    else:
        raise ValueError(f"Model {model_name} not supported")

Writing models.py


In [ ]:
%%writefile federated_utils.py
"""
Federated Learning utility functions
Aggregation, evaluation, and drift metrics
"""

import torch
import numpy as np
import copy


def get_model_weights(model):
    """
    Extract weights from a PyTorch model

    Args:
        model: PyTorch model

    Returns:
        list of tensors (detached and on CPU)
    """
    return [param.detach().cpu().clone() for param in model.parameters()]


def set_model_weights(model, weights):
    """
    Load weights into a PyTorch model

    Args:
        model: PyTorch model
        weights: list of tensors
    """
    with torch.no_grad():
        for param, weight in zip(model.parameters(), weights):
            param.copy_(weight.to(param.device))


def aggregate_weights(client_weights, client_sizes):
    """
    Weighted averaging of client model weights

    Args:
        client_weights: list of [weights for each client]
        client_sizes: list of number of samples for each client

    Returns:
        aggregated weights (list of tensors)
    """
    total_size = sum(client_sizes)

    # Initialize with zeros
    aggregated = [torch.zeros_like(client_weights[0][i]) for i in range(len(client_weights[0]))]

    # Weighted sum
    for client_weight, size in zip(client_weights, client_sizes):
        weight_factor = size / total_size
        for i, param in enumerate(client_weight):
            aggregated[i] += param * weight_factor

    return aggregated


def compute_weight_divergence(client_weights, global_weights):
    """
    Compute the average L2 distance between client models and global model

    Args:
        client_weights: list of [weights for each client]
        global_weights: global model weights

    Returns:
        average divergence (float)
    """
    divergences = []

    for client_weight in client_weights:
        divergence = 0.0
        for client_param, global_param in zip(client_weight, global_weights):
            divergence += torch.norm(client_param - global_param).item() ** 2
        divergences.append(np.sqrt(divergence))

    return np.mean(divergences)


def evaluate_model(model, test_loader, device='cpu', criterion=None):
    """
    Evaluate model on test set

    Args:
        model: PyTorch model
        test_loader: DataLoader for test set
        device: device to run evaluation on
        criterion: loss function (default: CrossEntropyLoss)

    Returns:
        accuracy, loss
    """
    if criterion is None:
        criterion = torch.nn.CrossEntropyLoss()

    model.eval()
    model.to(device)

    correct = 0
    total = 0
    total_loss = 0.0

    with torch.no_grad():
        for data, target in test_loader:
            data, target = data.to(device), target.to(device)
            output = model(data)

            # Loss
            loss = criterion(output, target)
            total_loss += loss.item() * data.size(0)

            # Accuracy
            _, predicted = output.max(1)
            total += target.size(0)
            correct += predicted.eq(target).sum().item()

    accuracy = 100.0 * correct / total
    avg_loss = total_loss / total

    return accuracy, avg_loss


def flatten_weights(weights):
    """
    Flatten a list of weight tensors into a single vector

    Args:
        weights: list of tensors

    Returns:
        flattened 1D tensor
    """
    return torch.cat([w.flatten() for w in weights])


def unflatten_weights(flat_weights, shapes):
    """
    Unflatten a 1D tensor back to list of tensors with given shapes

    Args:
        flat_weights: 1D tensor
        shapes: list of shapes for each parameter

    Returns:
        list of tensors
    """
    weights = []
    offset = 0
    for shape in shapes:
        numel = np.prod(shape)
        weights.append(flat_weights[offset:offset+numel].view(shape))
        offset += numel
    return weights


def compute_gradient_from_weights(initial_weights, updated_weights, lr):
    """
    Compute effective gradient from weight update
    gradient = (initial_weights - updated_weights) / lr

    Args:
        initial_weights: initial model weights
        updated_weights: updated model weights
        lr: learning rate used

    Returns:
        list of gradient tensors
    """
    gradients = []
    for init_w, upd_w in zip(initial_weights, updated_weights):
        grad = (init_w - upd_w) / lr
        gradients.append(grad)
    return gradients

Writing federated_utils.py


In [7]:
%%writefile client.py
"""
Client class for Federated Learning
Handles local training
"""

import torch
import torch.nn as nn
import torch.optim as optim
import copy
from scriptsfl.federated_utils import get_model_weights, set_model_weights


class Client:
    """
    Federated Learning Client

    Handles local training with vanilla SGD (can be extended for other methods)
    """

    def __init__(self, client_id, data_loader, model, device='cpu'):
        """
        Initialize client

        Args:
            client_id: unique client identifier
            data_loader: DataLoader for this client's data
            model: PyTorch model (will be copied)
            device: device to train on
        """
        self.client_id = client_id
        self.data_loader = data_loader
        self.device = device

        # Create a local copy of the model
        self.model = copy.deepcopy(model).to(device)

        # Data size
        self.data_size = len(data_loader.dataset)

    def train_local(self, epochs, lr, momentum=0.9):
        """
        Train locally using vanilla SGD

        Args:
            epochs: number of local epochs
            lr: learning rate
            momentum: SGD momentum

        Returns:
            dict with training stats
        """
        self.model.train()
        criterion = nn.CrossEntropyLoss()
        optimizer = optim.SGD(self.model.parameters(), lr=lr, momentum=momentum)

        epoch_losses = []

        for epoch in range(epochs):
            running_loss = 0.0
            for data, target in self.data_loader:
                data, target = data.to(self.device), target.to(self.device)

                optimizer.zero_grad()
                output = self.model(data)
                loss = criterion(output, target)
                loss.backward()
                optimizer.step()

                running_loss += loss.item() * data.size(0)

            epoch_loss = running_loss / self.data_size
            epoch_losses.append(epoch_loss)

        stats = {
            'client_id': self.client_id,
            'epochs': epochs,
            'final_loss': epoch_losses[-1],
            'epoch_losses': epoch_losses
        }

        return stats

    def get_weights(self):
        """
        Get current model weights

        Returns:
            list of tensors
        """
        return get_model_weights(self.model)

    def set_weights(self, weights):
        """
        Set model weights

        Args:
            weights: list of tensors
        """
        set_model_weights(self.model, weights)

    def get_data_size(self):
        """
        Get number of training samples

        Returns:
            int: number of samples
        """
        return self.data_size

Overwriting client.py


In [8]:
%%writefile server.py
"""
Server class for Federated Learning
Handles client selection, broadcasting, and aggregation
"""

import torch
import numpy as np
import copy
from scriptsfl.federated_utils import get_model_weights, set_model_weights, aggregate_weights, evaluate_model


class Server:
    """
    Federated Learning Server

    Orchestrates the federated training process
    """

    def __init__(self, global_model, clients, test_loader, device='cpu'):
        """
        Initialize server

        Args:
            global_model: PyTorch model
            clients: list of Client objects
            test_loader: DataLoader for test set
            device: device to run evaluation on
        """
        self.global_model = global_model.to(device)
        self.clients = clients
        self.test_loader = test_loader
        self.device = device

        # Training history
        self.history = {
            'rounds': [],
            'test_accuracy': [],
            'test_loss': [],
            'train_loss': [],
            'selected_clients': []
        }

    def select_clients(self, fraction=1.0, num_clients=None):
        """
        Randomly select clients for this round

        Args:
            fraction: fraction of clients to select (0 < fraction <= 1)
            num_clients: exact number of clients to select (overrides fraction)

        Returns:
            list of selected Client objects
        """
        if num_clients is None:
            num_clients = max(1, int(len(self.clients) * fraction))

        selected = np.random.choice(self.clients, size=num_clients, replace=False)
        return list(selected)

    def broadcast_weights(self, clients=None):
        """
        Send global model weights to clients

        Args:
            clients: list of clients (default: all clients)
        """
        if clients is None:
            clients = self.clients

        global_weights = get_model_weights(self.global_model)

        for client in clients:
            client.set_weights(global_weights)

    def aggregate(self, clients):
        """
        Aggregate client models into global model

        Args:
            clients: list of Client objects that participated

        Returns:
            average divergence of client models from global
        """
        # Get weights from all clients
        client_weights = [client.get_weights() for client in clients]
        client_sizes = [client.get_data_size() for client in clients]

        # Aggregate
        aggregated_weights = aggregate_weights(client_weights, client_sizes)

        # Update global model
        set_model_weights(self.global_model, aggregated_weights)

        # Compute divergence (optional, for analysis)
        from scriptsfl.federated_utils import compute_weight_divergence
        divergence = compute_weight_divergence(client_weights, aggregated_weights)

        return divergence

    def evaluate(self):
        """
        Evaluate global model on test set

        Returns:
            accuracy, loss
        """
        accuracy, loss = evaluate_model(self.global_model, self.test_loader, self.device)
        return accuracy, loss

    def train_round(self, local_epochs, lr, client_fraction=1.0, momentum=0.9):
        """
        Execute one round of federated training

        Args:
            local_epochs: number of local epochs per client
            lr: learning rate
            client_fraction: fraction of clients to select
            momentum: SGD momentum

        Returns:
            dict with round statistics
        """
        # Select clients
        selected_clients = self.select_clients(fraction=client_fraction)

        # Broadcast global model
        self.broadcast_weights(selected_clients)

        # Local training
        client_stats = []
        for client in selected_clients:
            stats = client.train_local(epochs=local_epochs, lr=lr, momentum=momentum)
            client_stats.append(stats)

        # Aggregate
        divergence = self.aggregate(selected_clients)

        # Evaluate global model
        test_acc, test_loss = self.evaluate()

        # Compute average training loss
        avg_train_loss = np.mean([s['final_loss'] for s in client_stats])

        round_stats = {
            'test_accuracy': test_acc,
            'test_loss': test_loss,
            'train_loss': avg_train_loss,
            'divergence': divergence,
            'num_clients': len(selected_clients),
            'client_ids': [c.client_id for c in selected_clients]
        }

        return round_stats

    def train(self, num_rounds, local_epochs, lr, client_fraction=1.0, momentum=0.9, verbose=True):
        """
        Train for multiple rounds

        Args:
            num_rounds: number of communication rounds
            local_epochs: number of local epochs per round
            lr: learning rate
            client_fraction: fraction of clients to select each round
            momentum: SGD momentum
            verbose: whether to print progress

        Returns:
            training history dict
        """
        for round_idx in range(num_rounds):
            round_stats = self.train_round(local_epochs, lr, client_fraction, momentum)

            # Update history
            self.history['rounds'].append(round_idx + 1)
            self.history['test_accuracy'].append(round_stats['test_accuracy'])
            self.history['test_loss'].append(round_stats['test_loss'])
            self.history['train_loss'].append(round_stats['train_loss'])
            self.history['selected_clients'].append(round_stats['client_ids'])

            if verbose:
                print(f"Round {round_idx + 1}/{num_rounds} | "
                      f"Test Acc: {round_stats['test_accuracy']:.2f}% | "
                      f"Test Loss: {round_stats['test_loss']:.4f} | "
                      f"Train Loss: {round_stats['train_loss']:.4f} | "
                      f"Divergence: {round_stats['divergence']:.4f}")

        return self.history

Overwriting server.py


In [ ]:
%%writefile results_utils.py
"""
Utilities for saving and loading experimental results
"""

import pickle
import json
import os
import matplotlib.pyplot as plt
import numpy as np


def save_results(results, filename, results_dir='./results'):
    """
    Save experimental results to pickle file

    Args:
        results: dict with experimental results
        filename: name of file (without extension)
        results_dir: directory to save results
    """
    os.makedirs(results_dir, exist_ok=True)
    filepath = os.path.join(results_dir, f"{filename}.pkl")

    with open(filepath, 'wb') as f:
        pickle.dump(results, f)

    print(f"Results saved to {filepath}")


def load_results(filename, results_dir='./results'):
    """
    Load experimental results from pickle file

    Args:
        filename: name of file (with or without .pkl extension)
        results_dir: directory containing results

    Returns:
        dict with results
    """
    if not filename.endswith('.pkl'):
        filename += '.pkl'

    filepath = os.path.join(results_dir, filename)

    with open(filepath, 'rb') as f:
        results = pickle.load(f)

    return results


def plot_comparison(results_dict, metric='test_accuracy', title=None, save_path=None):
    """
    Plot comparison of multiple experiments

    Args:
        results_dict: dict of {experiment_name: results}
        metric: metric to plot ('test_accuracy' or 'test_loss')
        title: plot title
        save_path: path to save figure (optional)
    """
    plt.figure(figsize=(10, 6))

    for exp_name, results in results_dict.items():
        if 'history' in results:
            history = results['history']
        else:
            history = results

        rounds = history.get('rounds', range(1, len(history[metric]) + 1))
        values = history[metric]

        plt.plot(rounds, values, marker='o', label=exp_name, linewidth=2)

    plt.xlabel('Communication Rounds', fontsize=12)

    if metric == 'test_accuracy':
        plt.ylabel('Test Accuracy (%)', fontsize=12)
        if title is None:
            title = 'Test Accuracy vs Communication Rounds'
    elif metric == 'test_loss':
        plt.ylabel('Test Loss', fontsize=12)
        if title is None:
            title = 'Test Loss vs Communication Rounds'

    plt.title(title, fontsize=14, fontweight='bold')
    plt.legend(fontsize=10)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f"Figure saved to {save_path}")

    plt.show()


def print_summary(results_dict):
    """
    Print summary statistics for multiple experiments

    Args:
        results_dict: dict of {experiment_name: results}
    """
    print("\n" + "="*80)
    print("EXPERIMENTAL RESULTS SUMMARY")
    print("="*80)

    for exp_name, results in results_dict.items():
        if 'history' in results:
            history = results['history']
        else:
            history = results

        final_acc = history['test_accuracy'][-1]
        best_acc = max(history['test_accuracy'])
        final_loss = history['test_loss'][-1]

        print(f"\n{exp_name}:")
        print(f"  Final Test Accuracy: {final_acc:.2f}%")
        print(f"  Best Test Accuracy:  {best_acc:.2f}%")
        print(f"  Final Test Loss:     {final_loss:.4f}")

        # Print configuration if available
        if 'config' in results:
            print(f"  Configuration:")
            for key, value in results['config'].items():
                print(f"    {key}: {value}")

    print("="*80 + "\n")

Writing results_utils.py
